# eco-movement data pipeline — end-to-end

This notebook walks through exactly what happens when the collector
pulls data from Mobilithek for the **eco-movement** provider, from the
raw HTTPS bytes to the rows that land in the SQLite databases used by
the dashboard.

**Sections**

1. Setup
2. **Static feed** — the infrastructure metadata (45k+ points)
   - 2.1 Raw bytes from Mobilithek
   - 2.2 Decoded JSON / DATEX II envelope
   - 2.3 Drilling down: table → site → station → refill point
   - 2.4 Parsed into a pandas DataFrame
   - 2.5 Written to the static SQLite DB
3. **Dynamic feed** — the live status stream
   - 3.1 Raw bytes (SNAPSHOT vs DELTA)
   - 3.2 Decoded JSON / status publication envelope
   - 3.3 A single point-status record
   - 3.4 Parsed into a DataFrame
   - 3.5 Written to the dynamic SQLite DB
   - 3.6 How the power estimate is derived
4. **What the dashboard reads**

Every database write in this notebook goes into a throw-away
`notebook_sandbox/` directory next to the notebook — the production
`data/` folder is not touched.


## 1  Setup

Imports, load the Mobilithek mTLS certificate from `.env`, and switch
the working directory to a sandbox so nothing we do here overwrites the
live databases.


In [ ]:
import os
import sys
import json
import gzip
import sqlite3
import shutil
from pathlib import Path
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv
from requests_pkcs12 import get as pkcs12_get

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 60)

# Project root (this notebook lives in the project root).
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "providers").is_dir(), "Run from the project root"

# Load the Mobilithek cert path + password.
load_dotenv(PROJECT_ROOT / ".env")
CERT_PATH = os.environ["MOBILITHEK_CERT_PATH"]
CERT_PASSWORD = os.environ["MOBILITHEK_CERT_PASSWORD"]
if not Path(CERT_PATH).is_absolute():
    CERT_PATH = str(PROJECT_ROOT / CERT_PATH)
# Normalise the env var too — the providers module re-reads it at
# import time to drive its own mTLS calls.
os.environ["MOBILITHEK_CERT_PATH"] = CERT_PATH
print("cert OK:", Path(CERT_PATH).exists())

# Sandbox: all SQLite files created by this notebook go here.
SANDBOX = PROJECT_ROOT / "notebook_sandbox"
shutil.rmtree(SANDBOX, ignore_errors=True)
(SANDBOX / "data").mkdir(parents=True)
os.chdir(SANDBOX)
print("cwd:", Path.cwd())

# Make the providers package importable.
sys.path.insert(0, str(PROJECT_ROOT))
from providers import get_provider  # noqa: E402
import providers.base as _pb  # noqa: E402
# Guarantee the provider sees the absolute cert path regardless of cwd.
_pb.CERT_PATH = CERT_PATH
import collector as col  # noqa: E402

provider = get_provider("eco_movement")
print("provider:", provider.name, "| slug:", provider.slug)
print("in_use_status:", provider.in_use_status)
print("tracked_columns:", provider.tracked_columns)
print("power_column:", provider.power_column, "(to MW:", provider.power_to_mw, ")")


## 2  Static feed — infrastructure metadata

This feed answers: *what chargers exist, where are they, and what
are their rated specs?*

It's a single, several-MB JSON document that Mobilithek re-publishes
whenever the upstream operator revises their inventory.


### 2.1  Raw bytes from Mobilithek

An HTTPS GET against the subscription URL, authenticated with the PKCS#12
client certificate. Mobilithek returns gzip-compressed JSON, carried
with a `Content-Encoding: gzip` header (which `requests` usually
decompresses for us).


In [ ]:
from providers.eco_movement import STATIC_URL, STATIC_SUBSCRIPTION_ID

print("subscription ID:", STATIC_SUBSCRIPTION_ID)
print("URL:", STATIC_URL)

resp = pkcs12_get(
    STATIC_URL,
    pkcs12_filename=CERT_PATH,
    pkcs12_password=CERT_PASSWORD,
    headers={"If-Modified-Since": "Thu, 01 Jan 1970 00:00:00 GMT"},
    timeout=180,
)
print("HTTP status:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))
print("Content-Encoding:", resp.headers.get("Content-Encoding"))
print("Last-Modified:", resp.headers.get("Last-Modified"))
print("body size (bytes):", len(resp.content))


The request library transparently un-gzipped the payload for us, so
`resp.content` is now readable bytes — but let's peek at what the very
first bytes look like. If the feed ever arrived *without* auto-gzip
handling (the push receiver, for instance, sees raw bytes) we would
see the gzip magic number `\x1f\x8b` at the start and have to
`gzip.decompress()` it ourselves.


In [ ]:
raw_bytes = resp.content
print("first 16 bytes (hex):", raw_bytes[:16].hex())
print("first 16 bytes (repr):", repr(raw_bytes[:16]))

# If still gzip-compressed, decompress.
if raw_bytes[:2] == b"\x1f\x8b":
    raw_bytes = gzip.decompress(raw_bytes)
    print("after gunzip:", len(raw_bytes), "bytes")

print()
print("first 400 chars:")
print(raw_bytes[:400].decode("utf-8"))


### 2.2  Decoded JSON — the DATEX II envelope

Mobilithek wraps the actual payload in the DATEX II *AFIR Energy
Infrastructure* schema (profile `AFIR Energy InfrastructureG`,
version 3.5 with German extensions). The outer object carries
version/profile metadata; the useful bit lives under
`payload → aegiEnergyInfrastructureTablePublication`.


In [ ]:
static_json = json.loads(raw_bytes)
print("top-level keys:", list(static_json.keys()))
print()
print("payload keys:", list(static_json["payload"].keys()))
print()
pub = static_json["payload"]["aegiEnergyInfrastructureTablePublication"]
print("publication keys:", list(pub.keys()))
print()
print("publicationTime:", pub.get("publicationTime"))
print("publicationCreator:", pub.get("publicationCreator"))
print("n tables:", len(pub.get("energyInfrastructureTable", [])))


### 2.3  Drilling down — table → site → station → refill point

The DATEX II hierarchy is four levels deep:

```
energyInfrastructureTable      (one per feed)
└── energyInfrastructureSite   (a location, e.g. a parking lot)
    └── energyInfrastructureStation   (a physical charger box)
        └── refillPoint            (an individual plug/EVSE)
```

Each level carries its own metadata (operator, coordinates, rated
power, connector type) that we fuse together into one flat row per
refill point in the DB.

Let's look at the first table, then the first site with stations, then
the first refill point.


In [ ]:
table = pub["energyInfrastructureTable"][0]
print("table keys:", list(table.keys()))
print("table id:", table.get("idG"))
print("n sites:", len(table.get("energyInfrastructureSite", [])))


In [ ]:
# Find the first site that actually has stations with refill points.
first_site = next(
    s for s in table["energyInfrastructureSite"]
    if s.get("energyInfrastructureStation")
    and s["energyInfrastructureStation"][0].get("refillPoint")
)

print("=== SITE ===")
print("id:", first_site.get("idG"))
print("name:", first_site.get("name"))
# Site-level location:
loc = first_site.get("locationReference", {})
pt = loc.get("locPointLocation", {})
coords = pt.get("coordinatesForDisplay", {})
print("lat/lon:", coords.get("latitude"), "/", coords.get("longitude"))
addr = pt.get("locLocationExtensionG", {}).get("facilityLocation", {}).get("address", {})
print("city:", addr.get("city"))
print("postcode:", addr.get("postcode"))
print("operator:",
      first_site.get("operator", {})
      .get("afacAnOrganisation", {})
      .get("name"))
print()
print("n stations at this site:", len(first_site["energyInfrastructureStation"]))


In [ ]:
station = first_site["energyInfrastructureStation"][0]
print("=== STATION ===")
print("id:", station.get("idG"))
print("totalMaximumPower (W):", station.get("totalMaximumPower"))
print("numberOfRefillPoints:", station.get("numberOfRefillPoints"))
print("authentication methods:",
      [a.get("value") for a in station.get("authenticationAndIdentificationMethods", [])])
print("n refill points at this station:", len(station.get("refillPoint", [])))


In [ ]:
# The refill point is wrapped in an "aegiElectricChargingPoint" dict.
rp = station["refillPoint"][0]
ecp = rp["aegiElectricChargingPoint"]
print("=== REFILL POINT (a.k.a. EVSE / charging point) ===")
print("id (point_id in the DB):", ecp.get("idG"))
print("currentType:", ecp.get("currentType"))
print("numberOfConnectors:", ecp.get("numberOfConnectors"))
print("availableChargingPower (W):", ecp.get("availableChargingPower"))
print()
print("connectors:")
for c in ecp.get("connector", []):
    print("  ", {
        "type": (c.get("connectorType") or {}).get("value"),
        "format": (c.get("connectorFormat") or {}).get("value"),
        "maxPowerAtSocket (W)": c.get("maxPowerAtSocket"),
    })
print()
# EVSE ID — the European cross-operator standard identifier.
for ext in ecp.get("externalIdentifier", []):
    toi = ext.get("typeOfIdentifier", {})
    if toi.get("extendedValueG") == "evseId":
        print("EVSE ID:", ext.get("identifier"))
        break


### 2.4  Parsed into a DataFrame

The provider's `_parse_static_points()` method walks the whole DATEX II
tree and flattens it into one row per refill point, picking out only
the fields we actually use downstream. Column types are normalised
(lat/lon/power → numeric), multilingual name objects are unwrapped to
a single string, and connector arrays are collapsed to a comma-joined
list.


In [ ]:
static_df = provider._parse_static_points(static_json)

print("shape:", static_df.shape)
print()
print("columns:")
print(list(static_df.columns))
print()
print("dtypes:")
print(static_df.dtypes)


In [ ]:
# Same site+station+point we inspected above, now as a single flat row:
same_id = ecp.get("idG")
same_row = static_df.loc[static_df["point_id"] == same_id].iloc[0]
print(same_row.to_string())


In [ ]:
# Fleet-wide stats: what the dashboard builds on.
print("total refill points:", len(static_df))
print("unique operators:", static_df["operator_name"].nunique())
print()
print("top operators by count:")
print(static_df["operator_name"].value_counts().head(8))
print()
print("rated power per point (kW):")
print((static_df["point_power_w"] / 1000).describe().round(1))
print()
print("fleet nameplate capacity: "
      f"{static_df['point_power_w'].sum() * 1e-6:,.0f} MW")


### 2.5  Written to the static SQLite DB

`ensure_static_db()` opens the provider's `{slug}_static.sqlite` file,
runs the `CREATE TABLE IF NOT EXISTS` defined on the provider, applies
any column migrations, and turns on WAL mode for concurrent reads.

`store_static_snapshot()` then deduplicates the DataFrame by
`point_id`, stamps a `fetched_at_utc` column, and `INSERT`s.

(Note: there is a known pre-existing bug where re-running this on a
populated DB fails with a `UNIQUE constraint` error — the notebook uses
a fresh sandbox DB so you will not see that here.)


In [ ]:
# Create a fresh static DB in the sandbox.
static_conn = col.ensure_static_db(provider)
print("DB path:", provider.static_db_path)

# Show the schema.
print("\n-- schema --")
for row in static_conn.execute(
    "SELECT sql FROM sqlite_master WHERE type='table'"
):
    print(row[0])


In [ ]:
# Publication time comes from the JSON envelope, not the HTTP header.
pub_time = pd.to_datetime(pub["publicationTime"], utc=True)
print("upstream publicationTime:", pub_time)

col.store_static_snapshot(static_conn, static_df, pub_time)

# Query back.
print("\nrows in charging_points:",
      static_conn.execute("SELECT COUNT(*) FROM charging_points").fetchone()[0])

print("\nstatic_meta:")
for row in static_conn.execute("SELECT key, value FROM static_meta"):
    print(" ", row[0], "=", row[1])


In [ ]:
# The DB version of our sample point — this is exactly what the
# dashboard sees when rendering the map and computing power estimates.
sample = pd.read_sql_query(
    "SELECT * FROM charging_points WHERE point_id = ?",
    static_conn, params=(same_id,),
)
print(sample.T.to_string())


## 3  Dynamic feed — live status

This feed answers: *which chargers are available / in-use / offline
right now?*

Mobilithek delivers this as a sequence of JSON messages, each of which
is either:

- **SNAPSHOT** — the complete state of every point. Arrives roughly
  once per day; serves as ground truth.
- **DELTA** — only the points whose status has changed since the last
  delivery. Arrives every few seconds.

The feed type is signalled by the HTTP `Type` header on each
response.


### 3.1  Raw bytes (SNAPSHOT vs DELTA)

`drain_dynamic_deliveries()` loops, pulling one delivery at a time and
advancing a `Last-Modified` cursor after each, until the server answers
`304 Not Modified` (queue empty).


In [ ]:
from providers.eco_movement import DYNAMIC_URL, DYNAMIC_SUBSCRIPTION_ID

print("subscription ID:", DYNAMIC_SUBSCRIPTION_ID)
print("URL:", DYNAMIC_URL)

# Fetch a single delivery directly, starting from the epoch so we
# definitely get the next queued message.
resp = pkcs12_get(
    DYNAMIC_URL,
    pkcs12_filename=CERT_PATH,
    pkcs12_password=CERT_PASSWORD,
    headers={"If-Modified-Since": "Thu, 01 Jan 1970 00:00:00 GMT"},
    timeout=120,
)
print("HTTP status:", resp.status_code)
print("Type (SNAPSHOT / DELTA):", resp.headers.get("Type"))
print("Last-Modified (cursor):", resp.headers.get("Last-Modified"))
print("Content-Type:", resp.headers.get("Content-Type"))
print("body size (bytes):", len(resp.content))


### 3.2  Decoded JSON — the status publication envelope

The dynamic payload uses a different top-level structure than the
static feed: everything is wrapped in a `messageContainer` that carries
exchange-protocol metadata, and the actual status data lives under
`messageContainer → payload[0] → aegiEnergyInfrastructureStatusPublication`.


In [ ]:
dyn_bytes = resp.content
if dyn_bytes[:2] == b"\x1f\x8b":
    dyn_bytes = gzip.decompress(dyn_bytes)

dynamic_json = json.loads(dyn_bytes)
print("top-level keys:", list(dynamic_json.keys()))

mc = dynamic_json["messageContainer"]
print("messageContainer keys:", list(mc.keys()))
print()
print("exchangeInformation:")
pprint(mc["exchangeInformation"], depth=3, width=100)


In [ ]:
# The actual status data.
status_pub = mc["payload"][0]["aegiEnergyInfrastructureStatusPublication"]
print("publicationTime:", status_pub.get("publicationTime"))
print("n site statuses in this delivery:",
      len(status_pub.get("energyInfrastructureSiteStatus", [])))

# Count individual point statuses (these roll up into sites → stations).
total_points = sum(
    len(stn.get("refillPointStatus", []))
    for site in status_pub.get("energyInfrastructureSiteStatus", [])
    for stn in site.get("energyInfrastructureStationStatus", [])
)
print("n point statuses in this delivery:", total_points)


### 3.3  A single point-status record

For each refill point the delivery carries its current `status` (one
of `available`, `charging`, `occupied`, `outOfOrder`, `unknown`, ...)
plus, optionally, price updates and queue-waiting time. The
`reference.idG` field is the EVSE ID we use to join back to the
static `charging_points` table.


In [ ]:
# Reach in for the first point status record.
first_site_status = status_pub["energyInfrastructureSiteStatus"][0]
first_station_status = first_site_status["energyInfrastructureStationStatus"][0]
first_rp_status = first_station_status["refillPointStatus"][0]

print("raw record:")
print(json.dumps(first_rp_status, indent=2, ensure_ascii=False)[:1500])


### 3.4  Parsed into a DataFrame

`parse_dynamic_points()` traverses the status tree and flattens it
into one row per point, carrying the columns declared in the
provider's `tracked_columns` (for eco-movement: `status`,
`price_per_kwh`, `waiting_time_s`).


In [ ]:
dyn_df, dyn_pub_time = provider.parse_dynamic_points(dynamic_json)
print("pub time:", dyn_pub_time)
print("rows:", len(dyn_df))

# Guard against intra-delivery duplicates (occasionally a point
# appears in more than one site-status block). We keep the last
# record seen, matching the behaviour of current_point_state.
before = len(dyn_df)
dyn_df = dyn_df.drop_duplicates(subset=["point_id"], keep="last")
if len(dyn_df) != before:
    print(f"deduped {before - len(dyn_df)} intra-delivery duplicates")

print()
print("status distribution:")
print(dyn_df["status"].value_counts(dropna=False))
print()
print("head:")
print(dyn_df.head(10).to_string(index=False))


### 3.5  Written to the dynamic SQLite DB

The dynamic DB has four tables:

| Table | Role |
|---|---|
| `snapshot_runs` | One row per delivery — aggregate counts + estimated power |
| `point_status_history` | One row per actual status *change* per point |
| `current_point_state` | Latest status per point — O(1) lookup for the dashboard |
| `dynamic_meta` | Cursor (last `Last-Modified`) for delta resumption |

`store_dynamic_snapshot()` does four things:

1. Diffs this delivery against the previous state and keeps only
   changed rows (no-op deltas are skipped entirely).
2. Rolls up the *full* updated state to compute counts.
3. Joins the in-use point IDs against the **static** DB to sum their
   rated power → `estimated_power_mw`.
4. Appends a row to `snapshot_runs` and changed rows to
   `point_status_history`; refreshes `current_point_state`.


In [ ]:
dynamic_conn = col.ensure_dynamic_db(provider)
print("DB path:", provider.dynamic_db_path)
print()
print("-- schema --")
for row in dynamic_conn.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' ORDER BY name"
):
    print(row[0], "\n")


In [ ]:
# Ingest the delivery we just pulled. Pass the static connection so
# estimated_power_mw can be computed.
snap_id, updated_state = col.store_dynamic_snapshot(
    dynamic_conn, provider, dyn_df, dyn_pub_time,
    delivery_type=resp.headers.get("Type", "SNAPSHOT"),
    previous_state=None,        # first ingest → full-snapshot semantics
    static_conn=static_conn,
)
print("inserted snapshot_id:", snap_id)
print("updated_state shape:", updated_state.shape)


In [ ]:
# The single snapshot_runs row we just wrote:
runs = pd.read_sql_query("SELECT * FROM snapshot_runs", dynamic_conn)
print(runs.T.to_string())


In [ ]:
# point_status_history — one row per change.
# Because this is the FIRST ingest (previous_state=None), EVERY point
# counts as changed, so we just inserted n_points history rows.
hist_n = dynamic_conn.execute(
    "SELECT COUNT(*) FROM point_status_history"
).fetchone()[0]
print("history rows:", hist_n)

hist = pd.read_sql_query(
    "SELECT snapshot_id, point_id, status, price_per_kwh, waiting_time_s"
    " FROM point_status_history LIMIT 5",
    dynamic_conn,
)
print(hist.to_string(index=False))


In [ ]:
# current_point_state — the dashboard's O(1) lookup table.
cps_n = dynamic_conn.execute(
    "SELECT COUNT(*) FROM current_point_state"
).fetchone()[0]
print("current_point_state rows:", cps_n)

sample_state = pd.read_sql_query(
    "SELECT * FROM current_point_state LIMIT 5", dynamic_conn,
)
print(sample_state.to_string(index=False))


Now let's drain a second delivery and watch how the DELTA path
behaves differently from a SNAPSHOT.


In [ ]:
# Advance the cursor using the Last-Modified header we captured.
col.set_cursor(dynamic_conn, resp.headers["Last-Modified"])

# Pull more deliveries through the provider's own drainer (uses the
# cursor in dynamic_meta).
deliveries = provider.drain_dynamic_deliveries(
    if_modified_since=col.get_cursor(dynamic_conn),
    max_deliveries=5,
)
print("pulled", len(deliveries), "additional deliveries")
for d in deliveries:
    print(" ", d["type"], "| Last-Modified:", d["last_modified"])


In [ ]:
# Ingest each of them. Each DELTA only affects the points that changed.
for d in deliveries:
    df_i, pt_i = provider.parse_dynamic_points(d["data"])
    if df_i.empty:
        continue
    df_i = df_i.drop_duplicates(subset=["point_id"], keep="last")
    if d["type"] == "SNAPSHOT":
        _, updated_state = col.store_dynamic_snapshot(
            dynamic_conn, provider, df_i, pt_i, "SNAPSHOT",
            previous_state=None, static_conn=static_conn,
        )
    else:
        _, updated_state = col.store_dynamic_snapshot(
            dynamic_conn, provider, df_i, pt_i, "DELTA",
            previous_state=updated_state, static_conn=static_conn,
        )

print()
runs_now = pd.read_sql_query(
    "SELECT snapshot_id, delivery_type, point_count,"
    " charging_count, available_count, estimated_power_mw"
    " FROM snapshot_runs ORDER BY snapshot_id",
    dynamic_conn,
)
print(runs_now.to_string(index=False))


Note how a DELTA delivery inserts a `snapshot_runs` row whose
`point_count` equals the total fleet (the *cumulative* state), not the
number of points in that particular delta message — that's the whole
point of the delta/full-state reconstruction.

The `point_status_history` table, on the other hand, grows only by the
number of rows that actually changed in each delta.


In [ ]:
hist_totals = pd.read_sql_query(
    "SELECT snapshot_id, COUNT(*) AS changed_rows"
    " FROM point_status_history GROUP BY snapshot_id ORDER BY snapshot_id",
    dynamic_conn,
)
print(hist_totals.to_string(index=False))


### 3.6  How the power estimate is derived

`estimated_power_mw` in `snapshot_runs` is the number the dashboard
plots. It's a simple join between the full state and the static
`point_power_w` column, summed over just the points whose current
status equals `provider.in_use_status` (for eco-movement:
`"charging"`).


In [ ]:
ATTACH_SQL = """
    SELECT SUM(cp.point_power_w) * 1e-6 AS mw
    FROM current_point_state cps
    JOIN static.charging_points cp USING (point_id)
    WHERE cps.status = ?
"""

# Briefly attach the static DB so we can join across the two files.
dynamic_conn.execute(
    f"ATTACH DATABASE '{provider.static_db_path}' AS static"
)
mw = dynamic_conn.execute(ATTACH_SQL, (provider.in_use_status,)).fetchone()[0]
print(f"current power draw ({provider.in_use_status}): {mw:.2f} MW")

dynamic_conn.execute("DETACH DATABASE static")


## 4  What the dashboard reads

Once `snapshot_runs` has hundreds or thousands of rows accumulated
(one per delivery, every few seconds), the dashboard just:

1. Reads `snapshot_runs` bucketed to 5-minute bins to plot the
   power-draw timeline.
2. Reads `current_point_state` joined with the static
   `charging_points` to render each charger as a dot on the map.
3. Uses SNAPSHOT rows as ground-truth anchors and interpolates a
   correction factor between them to compensate for delta drift.

That's the whole pipeline.


In [ ]:
# Clean up DB connections (sandbox files can be deleted after running).
static_conn.close()
dynamic_conn.close()
print("done. sandbox at:", Path.cwd())
